# Step 7: Feature Engineering

**SageMaker Unified Studio Component**: Feature Engineering

**What you'll learn**: Create features that improve model performance

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import boto3
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

account_id = boto3.client('sts').get_caller_identity()['Account']
bucket_name = os.getenv('BUCKET_NAME', f'sagemaker-unified-overheat-demo-{account_id}')
print(f"Using bucket: {bucket_name}")

## Load Cleaned Data

In [ ]:
s3_path = f's3://{bucket_name}/data/processed/clean_machines.parquet'
df = pd.read_parquet(s3_path)
print(f"Loaded {len(df):,} rows")
df.head()

## Create Temperature Difference Feature

In [ ]:
# Key feature: temperature difference from room
df['temp_diff'] = df['temperature'] - df['room_temp']

print("Created temp_diff feature")
print(f"Mean temp_diff: {df['temp_diff'].mean():.2f}°C")
print(f"Range: {df['temp_diff'].min():.2f} to {df['temp_diff'].max():.2f}°C")

## Create Target Variable

In [ ]:
# Label: overheat = temperature > 80
df['overheat'] = (df['temperature'] > 80).astype(int)

print("Created overheat label")
print(f"Overheating: {df['overheat'].sum():,} ({df['overheat'].mean()*100:.2f}%)")
print(f"Normal: {(~df['overheat'].astype(bool)).sum():,} ({(1-df['overheat'].mean())*100:.2f}%)")

## Visualize Feature Importance

In [ ]:
# Compare temp_diff for overheating vs normal
plt.figure(figsize=(10, 6))
df.boxplot(column='temp_diff', by='overheat')
plt.xlabel('Overheat (0=No, 1=Yes)')
plt.ylabel('Temperature Difference (°C)')
plt.title('Temperature Difference by Overheat Status')
plt.suptitle('')
plt.show()

print("Observation: Overheating machines have higher temp_diff")

## Create Final Feature Dataset

In [ ]:
# Select features and target
features = ['temperature', 'temp_diff']
target = 'overheat'

df_final = df[features + [target]]
df_final.head(10)

## Save Feature Dataset

In [ ]:
output_path = f's3://{bucket_name}/data/features/features.parquet'
df_final.to_parquet(output_path, index=False)
print(f"Saved to: {output_path}")

## Key Insights

**Why temp_diff works**:
- Machines normally run hotter than ambient
- Large difference indicates excessive heat generation
- More predictive than absolute temperature alone

**Next step**: Train the model